# bigdata8: Milestone2 联调测试与流处理阶段交付（Week08）

本 notebook 基于现有 `bigdata5-7.ipynb` 与 `m2_*.py` 代码，完成 week08 的集成目标：
1. 统一 CLI 编排入口（`run_pipeline.py`）。
2. Consumer 侧 `try-except` 容错并写入 `dead_letter.log`。
3. Chaos 异常注入与高负载压测。
4. 背压水位监控与指标落盘。

## 与任务书对齐说明

- 任务1（CLI 编排）：已在 `run_pipeline.py` 中使用 `argparse` 提供 `--qps --queue-limit --duration-sec ...`。
- 任务1（死信降级）：模型推理异常不会中断线程，异常事件会追加到 `dead_letter.log`。
- 任务2（Chaos Testing）：支持 `--chaos-rate` 小概率注入缺字段、错类型、坏时间戳、非字典载荷。
- 任务2（背压监控）：指标落盘到 `m2_backpressure_metrics.csv`，并按高低水位自动启停背压。

注意：本 notebook 只负责编排与实验执行，不撰写实验报告正文。

In [ ]:
from run_pipeline import M2Config, run_pipeline

M2Config()

## CLI 运行示例（终端）

```bash
python run_pipeline.py \
  --qps 1000 \
  --queue-limit 500 \
  --consumer-workers 2 \
  --consumer-process-sec 0.005 \
  --duration-sec 600 \
  --chaos-rate 0.01 \
  --output-csv m2_scored_events.csv \
  --metrics-csv m2_backpressure_metrics.csv \
  --dead-letter-log dead_letter.log
```

说明：`qps` 可设置为远高于消费能力以触发明显背压现象，便于联调观察。

In [ ]:
# Notebook 内快速联调（默认短时，避免误跑 10 分钟）
cfg = M2Config(
    qps=300,
    queue_limit=200,
    consumer_workers=2,
    consumer_process_sec=0.004,
    duration_sec=15,
    chaos_rate=0.01,
    output_csv='m2_scored_events.csv',
    metrics_csv='m2_backpressure_metrics.csv',
    dead_letter_log='dead_letter.log',
)

# 需要执行时取消注释
# summary = run_pipeline(cfg)
# summary

In [ ]:
import json
from pathlib import Path

dead_path = Path('dead_letter.log')
if dead_path.exists() and dead_path.stat().st_size > 0:
    samples = []
    with dead_path.open('r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if i >= 5:
                break
            samples.append(json.loads(line))
    samples
else:
    print('dead_letter.log 当前为空（说明短时运行未出现异常，或尚未执行 run_pipeline）。')

In [ ]:
import pandas as pd
from pathlib import Path

metrics_path = Path('m2_backpressure_metrics.csv')
if metrics_path.exists() and metrics_path.stat().st_size > 0:
    metrics_df = pd.read_csv(metrics_path)
    display(metrics_df.tail(10))
    print('rows =', len(metrics_df))
    print('max_queue_depth =', metrics_df['queue_depth'].max())
    print('max_load_pct =', metrics_df['load_pct'].max())
    print('backpressure_active_count =', metrics_df['backpressure_active'].sum())
else:
    print('m2_backpressure_metrics.csv 尚未生成。')

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

metrics_path = Path('m2_backpressure_metrics.csv')
if metrics_path.exists() and metrics_path.stat().st_size > 0:
    df = pd.read_csv(metrics_path)
    fig, ax1 = plt.subplots(figsize=(10, 4))

    ax1.plot(df['elapsed_sec'], df['queue_depth'], label='queue_depth', color='tab:blue')
    ax1.set_xlabel('elapsed_sec')
    ax1.set_ylabel('queue_depth', color='tab:blue')
    ax1.tick_params(axis='y', labelcolor='tab:blue')

    ax2 = ax1.twinx()
    ax2.plot(df['elapsed_sec'], df['load_pct'], label='load_pct', color='tab:orange', alpha=0.8)
    ax2.set_ylabel('load_pct', color='tab:orange')
    ax2.tick_params(axis='y', labelcolor='tab:orange')

    plt.title('Week08 M2 Queue Depth / Load Ratio')
    plt.tight_layout()
    plt.show()
else:
    print('先执行 run_pipeline 后再绘图。')

## 提交前自检建议

1. 进行一次长时压测（例如 `duration-sec=600`），确认背压触发与恢复日志符合预期。
2. 检查 `dead_letter.log` 是否持续记录异常事件，且主消费链路不中断。
3. 保留 `m2_backpressure_metrics.csv`、`m2_scored_events.csv` 作为报告数据来源。